# Test del modelo de segmentación de la pelota

In [27]:
#@title Download model and videos
import os
!pip install gdown # Install gdown
import gdown # Import gdown

FILE_ID_DATASET = '1ORO25C8ogbVsjrxtM6G7QDRZ61r7u10F'

FILE_ID_VIDEO_A2 = '1n25sh_iOss6n5RiD3OMDVt_Sd_7XTXwa'
FILE_ID_VIDEO_B2 = '1e-DoJcJNNwVDLHplUeHw4KvPR-K3oq4m'

FILE_ID_MODEL = '1XuxfD-Q3ffqu9vezvqhmj9W5EjtXS2QW'

models_dir = 'models'
os.makedirs(models_dir, exist_ok=True)
model_name = 'trained_yolo11_seg_ball.pt'
model_path = os.path.join(models_dir, model_name)

if not os.path.isfile(model_path):
  gdown.download(id=FILE_ID_MODEL, output=model_path, quiet=False)

if not os.path.isdir('videos'):
  os.makedirs('videos', exist_ok=True)
  gdown.download(id=FILE_ID_VIDEO_A2, output='videos/a2.mp4', quiet=False)
  gdown.download(id=FILE_ID_VIDEO_B2, output='videos/b2.mp4', quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1n25sh_iOss6n5RiD3OMDVt_Sd_7XTXwa
To: /content/videos/a2.mp4
100%|██████████| 15.3M/15.3M [00:00<00:00, 19.8MB/s]


'videos/a2.mp4'

In [28]:
!pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.3.234 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 38.4/112.6 GB disk)


In [29]:
import cv2

def segment_video(input_path, output_path, model, conf=-1, verbose=False):

  # Initialize video capture and video writer
  cap = cv2.VideoCapture(input_path)

  if not cap.isOpened():
      print("Error: Could not open input video.")
      return

  w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
  h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
  fps = int(cap.get(cv2.CAP_PROP_FPS))

  fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec for .mp4 files
  out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

  if verbose:
    print(f"Input video properties: Width={w}, Height={h}, FPS={fps}")
    print(f"Output video writer initialized for: {output_path}")

  # Process each frame
  while True:
      ret, frame = cap.read()
      if not ret:
          break

      # Process the frame with the YOLO model
      results = model.predict(frame, conf=conf, verbose=verbose) if conf>0 else model.predict(frame, verbose=verbose)

      # Draw the segmentation masks on the frame
      # results[0].plot() returns an annotated image (numpy array)
      annotated_frame = results[0].plot()

      # Write the annotated frame to the output video
      out.write(annotated_frame)

  # Release video capture and writer objects
  cap.release()
  out.release()

In [30]:
from ultralytics import YOLO
model = YOLO(model_path)

In [31]:
os.makedirs('results', exist_ok=True)

In [32]:
# @title Vídeo A
if True:
  segment_video(input_path='videos/a2.mp4', output_path='results/a2-seg.mp4', model=model)

In [41]:
# @title Vídeo B
if True:
  segment_video(input_path='videos/b2.mp4', output_path='results/b2-seg.mp4', model=model)

## Prueba con un vídeo de Youtube

In [34]:
!pip install pytubefix

In [35]:
from pytubefix import YouTube

def downloadYouTube(videourl, path, filename=None):
  YouTube(videourl).streams.get_highest_resolution().download(os.path.abspath(path), filename)

In [36]:
if False:
  downloadYouTube('https://www.youtube.com/watch?v=jgkDMsobvxg', './videos/sample', 'sample.mp4')

In [40]:
if False:
  segment_video(input_path='videos/sample/sample.mp4', output_path='results/sample-seg.mp4', model=model)
